In [1]:
import os
import re
import json
import pdfplumber

from PIL import Image
from transformers import pipeline
# from pipe_fn import pipe
from output_utils import save_split_output
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)


COLUMN_HEADERS = [
    "dos",
    "procedure_code",
    "billed_amount",
    "allowed_amount",
    "payable_amount",
    "copay_amount",
    "coins_amount",
    "deduct_amount",
    "overmax_amount",
    "patient_pay",
    "other_insur",
    "net_amount"
]

SCHEMA = {
    "rows": [
        {
            "provider": "",
            "dos": "",
            "procedure_code": "",
            "billed_amount": "",
            "allowed_amount": "",
            "payable_amount": "",
            "copay_amount": "",
            "coins_amount": "",
            "deduct_amount": "",
            "overmax_amount": "",
            "patient_pay": "",
            "other_insur": "",
            "net_amount": ""
        }
    ],
    "column_totals": {
        "billed_amount": "",
        "allowed_amount": "",
        "payable_amount": "",
        "copay_amount": "",
        "coins_amount": "",
        "deduct_amount": "",
        "overmax_amount": "",
        "patient_pay": "",
        "other_insur": "",
        "net_amount": ""
    }
}

# =========================================================
# EXTRACTION PROMPT
# =========================================================
prompt = """ You are extracting highly sensitive financial and dental claim table data from a PDF.

STRICT EXTRACTION RULES:

1. Extract ONLY table data.
2. Each row is independent and unique.
3. NEVER merge rows.
4. NEVER infer missing values from nearby rows.
5. NEVER copy values from adjacent columns.
6. Duplicate values are VALID financial data.
7. NEVER skip rows because values appear repeated.
8. Preserve row order exactly as shown in the PDF.
9. If a cell is empty, return "".
10. Extract values exactly from their respective columns only.

COLUMN MAPPING:
- Provider Name  -> provider
- DOS -> dos
- CODE -> procedure_code
- BILLED AMOUNT -> billed_amount
- ALLOWED AMOUNT -> allowed_amount
- PAYABLE AMOUNT -> payable_amount
- COPAY AMOUNT -> copay_amount
- COINS AMOUNT -> coins_amount
- DEDUCT AMOUNT -> deduct_amount
- OVER MAX AMOUNT -> overmax_amount
- PATIENT PAY -> patient_pay
- OTHER INSUR -> other_insur
- NET AMOUNT -> net_amount


HEADER EXTRACTION RULES:

1. provider
- Extract only the values after Provider Name: 

FIELD EXTRACTION RULES:

1. dos
- Extract only the date.
- Example:
  "02/25/26"

2. procedure_code
- Extract ONLY the first valid dental procedure code.
- Valid format:
  - Starts with uppercase "D"
  - Followed by exactly 4 digits
- Extract EXACTLY 5 characters only.
- Ignore anything after the first 5 characters.
- Examples:
  "D0150 00" -> "D0150"
  "D5110AB" -> "D5110"
  "D4559078" -> ""
  "D4532" -> "D4532"

3. Monetary columns
Extract only the value present inside that exact column:
- billed_amount
- allowed_amount
- payable_amount
- copay_amount
- coins_amount
- deduct_amount
- overmax_amount
- patient_pay
- other_insur
- net_amount

Rules:
- Do not borrow nearby values.
- Keep row alignment strict.
- Extract numeric/currency values exactly as shown.
- Example:
  "$1,319.00"

TOTAL ROW RULES:

1. A row WITHOUT both DOS and CODE is considered a totals row.
2. Totals row values must be stored inside:
   "column_totals"
3. Do NOT include totals row inside normal rows.
4. Totals rows usually contain summed financial values only.

OUTPUT FORMAT:

Return ONLY valid JSON.
No explanation.
No markdown.
No extra text.

IMPORTANT:
EVERY EXTRACTED FIELD MUST USE THIS FORMAT:

{
    "value": "",
    "confidence": 0.0
}

NEVER return an extracted field as a plain string.

For example, DO NOT return:

"procedure_code": "D0150"

ALWAYS return:

"procedure_code": {
    "value": "D0150",
    "confidence": 0.99
}

CONFIDENCE RULES:

- confidence MUST be a number between 0.0 and 1.0.
- 1.0 means completely certain.
- 0.0 means missing, unreadable, or cannot be reliably extracted.
- Confidence represents how certain you are that the extracted value is correct.
- Do NOT calculate confidence from the financial amount.
- Do NOT lower confidence simply because a value is zero.
- If a field is not visible or cannot be reliably extracted:

{
    "value": "",
    "confidence": 0.0
}

EVERY field in EVERY row MUST contain:
- value
- confidence

EVERY field in column_totals MUST contain:
- value
- confidence

FINAL JSON STRUCTURE:

{
    "provider": {
        "value": "",
        "confidence": 0.0
    },

    "rows": [
        {
            "dos": {
                "value": "",
                "confidence": 0.0
            },

            "procedure_code": {
                "value": "",
                "confidence": 0.0
            },

            "billed_amount": {
                "value": "",
                "confidence": 0.0
            },

            "allowed_amount": {
                "value": "",
                "confidence": 0.0
            },

            "payable_amount": {
                "value": "",
                "confidence": 0.0
            },

            "copay_amount": {
                "value": "",
                "confidence": 0.0
            },

            "coins_amount": {
                "value": "",
                "confidence": 0.0
            },

            "deduct_amount": {
                "value": "",
                "confidence": 0.0
            },

            "overmax_amount": {
                "value": "",
                "confidence": 0.0
            },

            "patient_pay": {
                "value": "",
                "confidence": 0.0
            },

            "other_insur": {
                "value": "",
                "confidence": 0.0
            },

            "net_amount": {
                "value": "",
                "confidence": 0.0
            }
        }
    ],

    "column_totals": {
        "billed_amount": {
            "value": "",
            "confidence": 0.0
        },

        "allowed_amount": {
            "value": "",
            "confidence": 0.0
        },

        "payable_amount": {
            "value": "",
            "confidence": 0.0
        },

        "copay_amount": {
            "value": "",
            "confidence": 0.0
        },

        "coins_amount": {
            "value": "",
            "confidence": 0.0
        },

        "deduct_amount": {
            "value": "",
            "confidence": 0.0
        },

        "overmax_amount": {
            "value": "",
            "confidence": 0.0
        },

        "patient_pay": {
            "value": "",
            "confidence": 0.0
        },

        "other_insur": {
            "value": "",
            "confidence": 0.0
        },

        "net_amount": {
            "value": "",
            "confidence": 0.0
        }
    }
}

FINAL CHECK:

Before returning the JSON, verify that EVERY extracted field contains:

"value"
"confidence"

If any field is a plain string, convert it to the required
value + confidence format.

Return ONLY JSON.

"""

# =========================================================
# IMAGE -> JSON EXTRACTION
# =========================================================

def extract_table_from_image(image_path):

    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    output = pipe(
        text=messages,
        max_new_tokens=2048
    )

    generated_text = output[0]["generated_text"]

    if isinstance(generated_text, list):
        generated_text = generated_text[-1]["content"]

    generated_text = generated_text.strip()

    generated_text = generated_text.replace(
        "```json", ""
    ).replace(
        "```", ""
    ).strip()

    return generated_text


# =========================================================
# ENFORCE SCHEMA
# =========================================================

def enforce_schema(parsed_output):

    allowed_columns = COLUMN_HEADERS

    for table in parsed_output.get("tables", []):

        cleaned_rows = []
        detected_totals = {}

        for row in table.get("rows", []):

            cleaned_row = {}

            for col in allowed_columns:

                value = row.get(col, "")

                if value is None:
                    value = ""

                value = str(value).strip()

                if col == "dos":
                    date_match = re.search(
                        r"\d{2}/\d{2}/\d{2,4}",
                        value
                    )
                    value = date_match.group(0) if date_match else ""

                elif col == "procedure_code":
                    match = re.match(r"(D\d{4})(?!\d)", value)
                    value = match.group(1) if match else ""

                else:
                    money_match = re.search(r"\$?[\d,]+\.\d{2}", value)
                    value = money_match.group(0) if money_match else ""

                cleaned_row[col] = value

            is_total_row = (
                cleaned_row["dos"] == ""
                and cleaned_row["procedure_code"] == ""
            )

            if is_total_row:
                for col in allowed_columns:
                    if col not in ["dos", "procedure_code"]:
                        val = cleaned_row.get(col, "")
                        if val != "":
                            detected_totals[col] = val
                continue

            if any(v not in ["", None] for v in cleaned_row.values()):
                cleaned_rows.append(cleaned_row)

        table["rows"] = cleaned_rows

        final_totals = {}
        existing_totals = table.get("column_totals", {})

        for col in allowed_columns:
            if col not in ["dos", "procedure_code"]:
                final_totals[col] = detected_totals.get(
                    col, existing_totals.get(col, "")
                )

        table["column_totals"] = final_totals

    return parsed_output


# =========================================================
# DENIAL CHECK — search within table region + ITEM notes
# =========================================================

def check_denial_in_region(page, start_y, end_y):

    DENIAL_KEYWORDS = ["denied", "denial"]

    # Extend to bottom of page by default
    # Shrink only if another Patient Name exists below end_y
    extended_end_y = page.height

    next_patient_hits = page.search("Patient Name", case=False)
    for hit in next_patient_hits:
        hit_y = float(hit["top"])
        if hit_y > end_y:
            extended_end_y = hit_y
            break

    print(f"    Denial search: y={start_y:.1f} → {extended_end_y:.1f}")

    for keyword in DENIAL_KEYWORDS:
        hits = page.search(
            rf"\b{keyword}\b",
            case=False,
            regex=True
        )
        for hit in hits:
            y = float(hit["top"])
            if start_y <= y <= extended_end_y:
                print(f"    🔴 '{keyword}' found at y={y:.1f}")
                return True

    return False


# =========================================================
# EXTRACT PATIENT NAME
# =========================================================

def extract_patient_name(page):

    words = page.extract_words()

    stop_words = {
        "Provider",
        "Subscriber/Member",
        "DOB",
        "Office",
        "Pay",
        "Encounter",
        "Referral",
        "Benefit",
        "Plan",
        "Product"
    }

    for i, w in enumerate(words):
        text = w["text"].strip()

        if (
            text == "Patient"
            and i + 1 < len(words)
            and "Name" in words[i + 1]["text"]
        ):
            name_parts = []

            for j in range(i + 2, i + 10):
                if j >= len(words):
                    break

                next_word = words[j]["text"].strip()

                if ":" in next_word or next_word in stop_words:
                    break

                name_parts.append(next_word)

            return " ".join(name_parts).strip()

    return ""


# =========================================================
# EXTRACT PATIENT NAME AT SPECIFIC Y POSITION
# =========================================================

def extract_patient_name_at_y(page, start_y, end_y):
    """Extract patient name from a specific region of the page."""

    words = page.extract_words()

    stop_words = {
        "Provider",
        "Subscriber/Member",
        "DOB",
        "Office",
        "Pay",
        "Encounter",
        "Referral",
        "Benefit",
        "Plan",
        "Product"
    }

    for i, w in enumerate(words):

        # Only look within this segment's y range
        if not (start_y <= float(w["top"]) <= end_y):
            continue

        text = w["text"].strip()

        if (
            text == "Patient"
            and i + 1 < len(words)
            and "Name" in words[i + 1]["text"]
        ):
            name_parts = []

            for j in range(i + 2, i + 10):
                if j >= len(words):
                    break

                next_word = words[j]["text"].strip()

                if ":" in next_word or next_word in stop_words:
                    break

                name_parts.append(next_word)

            return " ".join(name_parts).strip()

    return ""


# =========================================================
# COUNT SERVICE ROWS
# =========================================================

def count_service_rows(page, region_top, region_bottom):

    words = page.extract_words()
    row_positions = []

    for w in words:
        text = w["text"].strip()
        match = re.search(r"\bD\d{4}\b", text)

        if match:
            y = float(w["top"])
            if region_top <= y <= region_bottom:
                row_positions.append(y)

    row_positions.sort()

    grouped_rows = []
    threshold = 3

    for y in row_positions:
        if not grouped_rows:
            grouped_rows.append(y)
        else:
            if abs(y - grouped_rows[-1]) > threshold:
                grouped_rows.append(y)

    return len(grouped_rows)


# =========================================================
# VALIDATE EOB TABLE
# =========================================================

def parse_amount(val):
    if val in ["", None]:
        return 0.0
    return float(str(val).replace("$", "").replace(",", "").strip())


def validate_eob_table(table: dict, table_index: int):

    rows = table.get("rows", [])
    totals = table.get("column_totals", {})

    if not rows:
        return False, "", [], 0

    computed_totals = {
        "billed_amount":  round(sum(parse_amount(r.get("billed_amount", ""))  for r in rows), 2),
        "allowed_amount": round(sum(parse_amount(r.get("allowed_amount", "")) for r in rows), 2),
        "payable_amount": round(sum(parse_amount(r.get("payable_amount", "")) for r in rows), 2),
        "copay_amount":   round(sum(parse_amount(r.get("copay_amount", ""))   for r in rows), 2),
        "coins_amount":   round(sum(parse_amount(r.get("coins_amount", ""))   for r in rows), 2),
        "deduct_amount":  round(sum(parse_amount(r.get("deduct_amount", ""))  for r in rows), 2),
        "overmax_amount": round(sum(parse_amount(r.get("overmax_amount", "")) for r in rows), 2),
        "patient_pay":    round(sum(parse_amount(r.get("patient_pay", ""))    for r in rows), 2),
        "other_insur":    round(sum(parse_amount(r.get("other_insur", ""))    for r in rows), 2),
        "net_amount":     round(sum(parse_amount(r.get("net_amount", ""))     for r in rows), 2),
    }

    total_fields = len(computed_totals) 

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [Table {table_index}]")
    print("-" * 75)

    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(field, "")), 2)

        if computed_value == extracted_value:
            icon = "✅"
            status = "match"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True
            errors.append({
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    print("-" * 75)

    if has_error:
        print(f"❌ [Table {table_index}] Validation FAILED\n")
        return False, result_validation, errors, total_fields
    else:
        print(f"✅ [Table {table_index}] Validation PASSED\n")
        return True, result_validation, [], total_fields


def validate_service_row_count(page, start_y, end_y, table, table_index):

    detected_count = count_service_rows(page, start_y, end_y)

    rows = table.get("rows", [])
    extracted_count = len([
        r for r in rows
        if r.get("procedure_code") not in ["", None]
    ])

    print(f"\n📊 Row Count Validation [Table {table_index}]")
    print("-" * 70)

    icon   = "✅" if detected_count == extracted_count else "❌"
    status = "match" if detected_count == extracted_count else "MISMATCH"

    print(f"{icon} row_count detected={detected_count:<5} | extracted={extracted_count:<5} {status}")
    print("-" * 70)

    return detected_count == extracted_count


# =========================================================
# MAIN PIPELINE
# =========================================================

def crop_all_eob_tables(pdf_path, output_dir="EOB_OUTPUT/UHC_Government", company_name= "UHC government"):

    pdf_dir = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path)

    output_dir = os.path.join(output_dir, pdf_dir)
    os.makedirs(output_dir, exist_ok=True)

    all_patients=[]
    confidence_results = []

    claim_denied = False

    # =========================================================
    # PASS 1: Scan all pages, collect all segments
    # =========================================================
    with pdfplumber.open(pdf_path) as pdf:

        all_segments = []

        for page_num, page in enumerate(pdf.pages, start=1):

            print(f"\nScanning Page {page_num}")

            # -------------------------------------------------
            # Per-page denial keyword check
            # -------------------------------------------------
            DENIAL_KEYWORDS = ["denied", "denial"]
            page_denied = False

            for keyword in DENIAL_KEYWORDS:
                hits = page.search(
                    rf"\b{keyword}\b",
                    case=False,
                    regex=True
                )
                if hits:
                    page_denied = True
                    break

            print(f"  Page {page_num} — denial keyword found: {page_denied}")

            # -------------------------------------------------
            # Find Patient Name positions (table starts)
            # -------------------------------------------------
            patient_hits = page.search("Patient Name", case=False)
            if not patient_hits:
                patient_hits = page.search("Patient Name", case=False)

            # -------------------------------------------------
            # Find ITEM positions (table ends)
            # -------------------------------------------------
            item_hits = page.search("ITEM: 1", case=False)
            if not item_hits:
                item_hits = page.search("ITEM", case=False)

            patient_tops = sorted([
                max(0, h["top"] - 2) for h in patient_hits
            ])
            item_bottoms = sorted([
                h["bottom"] - 8 for h in item_hits
            ])

            print(f"  patient_tops = {[f'{y:.1f}' for y in patient_tops]}")
            print(f"  item_bottoms = {[f'{y:.1f}' for y in item_bottoms]}")

            # -------------------------------------------------
            # Match each Patient Name to its end boundary
            # Priority:
            #   1. First ITEM below this patient AND above next patient
            #   2. Next Patient Name on same page (no ITEM needed)
            #   3. End of page → continuation to next page
            # -------------------------------------------------
            used_items = set()

            for p_idx, p_top in enumerate(patient_tops):

                # Next patient on same page = upper boundary for ITEM search
                if p_idx + 1 < len(patient_tops):
                    next_patient_y = patient_tops[p_idx + 1]
                else:
                    next_patient_y = None

                # Find first ITEM below this patient AND above next patient
                paired_item = None
                for i_idx, item_y in enumerate(item_bottoms):
                    if i_idx not in used_items and item_y > p_top:
                        if next_patient_y is None or item_y <= next_patient_y:
                            paired_item = item_y
                            used_items.add(i_idx)
                            break

                if paired_item is not None:
                    # ✅ Ended by ITEM
                    end_y = paired_item
                    has_end = True
                    print(f"  ✅ Patient@{p_top:.1f} → ITEM end@{end_y:.1f}")

                elif next_patient_y is not None:
                    # ✅ Ended by next Patient Name (no ITEM between them)
                    end_y = next_patient_y
                    has_end = True
                    print(f"  ✅ Patient@{p_top:.1f} → next Patient@{end_y:.1f} (no ITEM)")

                else:
                    # ⏳ No end found on this page → continues to next page
                    end_y = page.height
                    has_end = False
                    print(f"  ⏳ Patient@{p_top:.1f} → page end (CONTINUING to next page)")

                seg = {
                    "page_num": page_num,
                    "page_obj": page,
                    "start_y":  p_top,
                    "end_y":    end_y,
                    "has_end":  has_end,
                }
                all_segments.append(seg)

           # -------------------------------------------------
            # Unmatched ITEMs = continuation from previous page
            # ONLY add if there are pending segments waiting
            # -------------------------------------------------
            unmatched_items = [
                item_bottoms[i]
                for i in range(len(item_bottoms))
                if i not in used_items
            ]

            # Check if previous pages have pending (no-end) segments
            has_pending = any(
                not s.get("has_end", True) and not s.get("is_continuation", False)
                for s in all_segments
            )

            for item_y in unmatched_items:

                if not has_pending:
                    # No pending table → this ITEM is orphan notes, SKIP
                    print(f"  ⚠️ Skipping orphan ITEM notes at y={item_y:.1f} — no pending table")
                    continue

                seg = {
                    "page_num":        page_num,
                    "page_obj":        page,
                    "start_y":         0,
                    "end_y":           item_y,
                    "has_end":         True,
                    "is_continuation": True,
                }
                all_segments.append(seg)
                print(f"  🔗 Continuation segment on page {page_num}: end_y={item_y:.1f}")

        # =========================================================
        # PASS 2: Group segments into logical tables
        # Rules:
        #   - has_end=False  → pending, wait for continuation
        #   - is_continuation → closes pending
        #   - Same patient name on consecutive pages → merge
        # =========================================================

        logical_tables = []
        pending = []

        all_segments.sort(key=lambda s: (s["page_num"], s["start_y"]))

        for seg in all_segments:

            is_continuation = seg.get("is_continuation", False)

            if is_continuation:
                # Closes whatever is pending
                if pending:
                    pending.append(seg)
                    logical_tables.append(list(pending))
                    pending = []
                else:
                    # Orphan continuation
                    logical_tables.append([seg])

            elif not seg["has_end"]:
                # Starts new table, no end yet
                if pending:
                    logical_tables.append(list(pending))
                    pending = []
                pending.append(seg)

            else:
                # Complete segment
                if pending:
                    # Check if same patient → merge
                    pending_name = extract_patient_name_at_y(
                        pending[0]["page_obj"],
                        pending[0]["start_y"],
                        pending[0]["end_y"]
                    )
                    current_name = extract_patient_name_at_y(
                        seg["page_obj"],
                        seg["start_y"],
                        seg["end_y"]
                    )

                    if (
                        pending_name
                        and current_name
                        and pending_name.strip() == current_name.strip()
                    ):
                        # Same patient → merge into one logical table
                        pending.append(seg)
                        logical_tables.append(list(pending))
                        pending = []
                        print(f"  🔗 Merged same patient: '{current_name}'")
                    else:
                        # Different patient → flush pending, save current separately
                        logical_tables.append(list(pending))
                        pending = []
                        logical_tables.append([seg])
                else:
                    logical_tables.append([seg])

        # Flush any remaining pending
        if pending:
            logical_tables.append(list(pending))

        print(f"\nTotal logical tables detected: {len(logical_tables)}")

        # =========================================================
        # PASS 3: Crop + stitch + extract for each logical table
        # =========================================================

        global_table_idx = 0


        for lt_idx, segments in enumerate(logical_tables, start=1):

            print(f"\n--- Logical Table {lt_idx} ({len(segments)} segment(s)) ---")

            cropped_images = []

            for seg in segments:

                page     = seg["page_obj"]
                start_y  = seg["start_y"]
                end_y    = seg["end_y"]
                page_num = seg["page_num"]

                if start_y >= end_y:
                    print(f"  Skipping invalid bbox on page {page_num}")
                    continue

                bbox    = (0, start_y, page.width, end_y)
                cropped = page.crop(bbox)
                pil_img = cropped.to_image(resolution=300).original
                cropped_images.append((page_num, pil_img))

                print(f"  Cropped page {page_num}: y={start_y:.1f}→{end_y:.1f}, size={pil_img.size}")

            if not cropped_images:
                print(f"  No valid crops for logical table {lt_idx}, skipping.")
                continue

            # -------------------------------------------------
            # Stitch all crops vertically
            # -------------------------------------------------
            if len(cropped_images) == 1:
                stitched = cropped_images[0][1]
            else:
                total_width  = max(img.width  for _, img in cropped_images)
                total_height = sum(img.height for _, img in cropped_images)
                stitched = Image.new("RGB", (total_width, total_height), color=(255, 255, 255))
                y_offset = 0
                for _, img in cropped_images:
                    stitched.paste(img, (0, y_offset))
                    y_offset += img.height

            global_table_idx += 1
            image_path = os.path.join(output_dir, f"table_{global_table_idx}.png")
            stitched.save(image_path)
            print(f"  Saved stitched image → {image_path}")

            first_seg = segments[0]

            try:
                # LLM extraction
                llm_output = extract_table_from_image(image_path)
                llm_output = (
                    llm_output
                    .replace("```json", "")
                    .replace("```", "")
                    .strip()
                )

                parsed_output = json.loads(llm_output)

                if "tables" not in parsed_output:
                    parsed_output = {"tables": [parsed_output]}


                # =========================================================
                # CONFIDENCE CALCULATION + UNWRAP
                # =========================================================

                for i, table in enumerate(parsed_output.get("tables", [])):

                    # Calculate confidence while the model output
                    # still contains {value, confidence}
                    table_model_confidence = calculate_model_confidence(table)

                    # Remove confidence wrapper:
                    # {"value": "D0150", "confidence": 0.99}
                    # becomes:
                    # "D0150"
                    table = _unwrap_vlm_output(table)

                    # Keep the overall model confidence internally
                    table["_model_confidence"] = table_model_confidence

                    # Replace the table
                    parsed_output["tables"][i] = table

                    


                # =========================================================
                # EXISTING SCHEMA CLEANING
                # =========================================================

                parsed_output = enforce_schema(parsed_output)
                # =========================================================
                # ROW COUNT ENFORCEMENT
                # =========================================================

                for table in parsed_output.get("tables", []):

                    expected_rows = 0

                    for seg in segments:
                        expected_rows += count_service_rows(
                            seg["page_obj"],
                            seg["start_y"],
                            seg["end_y"]
                        )

                    rows = table.get("rows", [])

                    extracted_count = len([
                        r for r in rows
                        if r.get("procedure_code") not in ["", None]
                    ])

                    print(
                        f"Expected Rows={expected_rows} | "
                        f"Extracted Rows={extracted_count}"
                    )

                    if extracted_count > expected_rows:

                        print(
                            f"⚠️ Extra rows detected: "
                            f"{extracted_count} -> {expected_rows}"
                        )

                        rows = rows[:expected_rows]

                        table["rows"] = rows


                # Patient name from first segment
                patient_name = extract_patient_name_at_y(
                    first_seg["page_obj"],
                    first_seg["start_y"],
                    first_seg["end_y"]
                )

                for table in parsed_output.get("tables", []):
                    saved_model_confidence = table.get("_model_confidence", 0.0)
                    new_table = {
                        "patient_name": patient_name,
                        "provider": table.get("provider", ""),
                        "EOB_ID":       pdf_dir,
                        "rows":         table.get("rows", []),
                        "column_totals": table.get("column_totals", {}),
                        "_model_confidence": saved_model_confidence,

                    }
                    table.clear()
                    table.update(new_table)

                
                    print(f"\n📊 Row Count Validation [Logical Table {lt_idx}]")
                    print("-" * 70)
                    icon   = "✅" if expected_rows == extracted_count else "❌"
                    status = "match" if expected_rows == extracted_count else "MISMATCH"

                    print(
                        f"{icon} row_count expected={expected_rows:<5} | "
                        f"extracted={extracted_count:<5} {status}"
                    )

                    print("-" * 70)

                # Amount validation
                # Amount validation
                is_valid = True
                validation_errors = []

                for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):
                    table_valid, log, table_errors, total_fields = validate_eob_table(table, t_idx)
                    if not table_valid:
                        is_valid = False
                        validation_errors.extend(table_errors)

                # Denial check across all segments
                is_denied = False
                for seg in segments:
                    if check_denial_in_region(
                        seg["page_obj"],
                        seg["start_y"],
                        seg["end_y"]
                    ):
                        is_denied = True
                        break

                print(f"\n  {'🔴 DENIED' if is_denied else '🟢 NOT DENIED'} — Logical Table {lt_idx}")
                if is_denied:
                    claim_denied = True

                # Save to final output
                for table in parsed_output.get("tables", []):
                    structured_table = {
                        "page":          first_seg["page_num"],
                        "table":         global_table_idx,
                        "patient_name":  table.get("patient_name", ""),
                        "provider": table.get("provider", ""),
                        "EOB_ID":        pdf_dir,
                        "denial_status": "Denied" if is_denied else "Not Denied",
                        "rows":          table.get("rows", []),
                        "column_totals": table.get("column_totals", {}),
                        "validation": {"status": is_valid, "errors": validation_errors},
                        "_expected_rows": expected_rows,
                        "_total_fields": total_fields,

                    }

                date_of_service = ""
                provider = ""

                if structured_table.get("rows"):
                    date_of_service = structured_table["rows"][0].get("dos", "")
                    # provider = structured_table["rows"][0].get("provider", "")

                for row in structured_table.get("rows", []):

                    for col in ["dos"]:
                        row.pop(col, None)


                patient_data = {
                    "patient_name": structured_table.get("patient_name", ""),
                    "provider": structured_table.get("provider", ""),
                    "date_of_service": date_of_service,
                    "services": structured_table.get("rows", []),
                    "totals": structured_table.get("column_totals", {}),
                    "validation": {"status": is_valid, "errors": validation_errors},
                    "_expected_rows": structured_table.get("_expected_rows", 0),
                    "_total_fields": structured_table.get("_total_fields", 0),
                    "_model_confidence": table.get("_model_confidence", 0.0),
                }                    

                all_patients.append(patient_data)
                confidence_results.append(patient_data)


            except json.JSONDecodeError as e:
                print(f"\nJSON Decode Error on logical table {lt_idx}: {e}")
            except Exception as e:
                import traceback

                print(f"\nError on logical table {lt_idx}: {e}")
                print("Full Traceback")
                traceback.print_exc()

    confidence_score = calculate_eob_confidence(confidence_results)


    final_output = [
        {
            "eob_id": pdf_dir,
            "file_name":pdf_full_name,
            "claim_status": "Denied" if claim_denied else "Not Denied",
            "payor": "UNITED HEALTHCARE - G",
            "confidence_score": confidence_score,
            "patients": all_patients
        }
    ]

    success_path, failed_path = save_split_output(
                                    final_output,
                                    company_name=company_name,
                                    pdf_name=pdf_dir,
                                    pdf_path=pdf_path,
                                    cropped_dir=output_dir,
                                )
                            
    print(f"\n📁 Cropped images : {output_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final_output

W0901 19:35:22.048000 3561429 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:35:22.062000 3561429 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
crop_all_eob_tables(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/United_health/497051284.pdf")


Scanning Page 1
  Page 1 — denial keyword found: False
  patient_tops = []
  item_bottoms = []

Scanning Page 2
  Page 2 — denial keyword found: False
  patient_tops = []
  item_bottoms = []

Scanning Page 3
  Page 3 — denial keyword found: False
  patient_tops = ['204.2', '400.1', '580.2']
  item_bottoms = ['347.0']
  ✅ Patient@204.2 → ITEM end@347.0
  ✅ Patient@400.1 → next Patient@580.2 (no ITEM)
  ⏳ Patient@580.2 → page end (CONTINUING to next page)

Scanning Page 4
  Page 4 — denial keyword found: True
  patient_tops = ['124.7', '441.8', '583.9']
  item_bottoms = ['557.8', '713.8', '727.3']
  ✅ Patient@124.7 → next Patient@441.8 (no ITEM)
  ✅ Patient@441.8 → ITEM end@557.8
  ✅ Patient@583.9 → ITEM end@713.8
  🔗 Continuation segment on page 4: end_y=727.3

Scanning Page 5
  Page 5 — denial keyword found: False
  patient_tops = ['132.2']
  item_bottoms = ['275.0']
  ✅ Patient@132.2 → ITEM end@275.0

Scanning Page 6
  Page 6 — denial keyword found: True
  patient_tops = []
  item_bo

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  Saved stitched image → EOB_OUTPUT/UHC_Government/497051284/table_1.png


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Expected Rows=3 | Extracted Rows=3

📊 Row Count Validation [Logical Table 1]
----------------------------------------------------------------------
✅ row_count expected=3     | extracted=3     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ billed_amount             computed=372.59     | extracted=372.59     match
✅ allowed_amount            computed=0.0        | extracted=0.0        match
✅ payable_amount            computed=0.0        | extracted=0.0        match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ coins_amount              computed=0.0        | extracted=0.0        match
✅ deduct_amount             computed=0.0        | extracted=0.0        match
✅ overmax_amount            computed=0.0        | extracted=0.0        match
✅ patient_pay               computed=0.0        | extracted=0.0        match
✅ other_in

...

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Expected Rows=1 | Extracted Rows=1

📊 Row Count Validation [Logical Table 5]
----------------------------------------------------------------------
✅ row_count expected=1     | extracted=1     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ billed_amount             computed=112.34     | extracted=112.34     match
✅ allowed_amount            computed=73.0       | extracted=73.0       match
✅ payable_amount            computed=73.0       | extracted=73.0       match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ coins_amount              computed=0.0        | extracted=0.0        match
✅ deduct_amount             computed=0.0        | extracted=0.0        match
✅ overmax_amount            computed=0.0        | extracted=0.0        match
✅ patient_pay               computed=0.0        | extracted=0.0        match
✅ other_in

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Expected Rows=2 | Extracted Rows=2

📊 Row Count Validation [Logical Table 6]
----------------------------------------------------------------------
✅ row_count expected=2     | extracted=2     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ billed_amount             computed=2284.57    | extracted=2284.57    match
✅ allowed_amount            computed=0.0        | extracted=0.0        match
✅ payable_amount            computed=0.0        | extracted=0.0        match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ coins_amount              computed=0.0        | extracted=0.0        match
✅ deduct_amount             computed=0.0        | extracted=0.0        match
✅ overmax_amount            computed=0.0        | extracted=0.0        match
✅ patient_pay               computed=891.0      | extracted=891.0      match
✅ other_in

[{'eob_id': '497051284',
  'file_name': '497051284.pdf',
  'claim_status': 'Denied',
  'payor': 'UNITED HEALTHCARE - G',
  'confidence_score': 99.5,
  'patients': [{'patient_name': 'HENRY, MIHA',
    'provider': 'Duc Tang',
    'date_of_service': '05/23/24',
    'services': [{'procedure_code': 'D1110',
      'billed_amount': '64.00',
      'allowed_amount': '0.00',
      'payable_amount': '0.00',
      'copay_amount': '0.00',
      'coins_amount': '0.00',
      'deduct_amount': '0.00',
      'overmax_amount': '0.00',
      'patient_pay': '0.00',
      'other_insur': '0.00',
      'net_amount': '0.00'},
     {'procedure_code': 'D0367',
      'billed_amount': '249.59',
      'allowed_amount': '0.00',
      'payable_amount': '0.00',
      'copay_amount': '0.00',
      'coins_amount': '0.00',
      'deduct_amount': '0.00',
      'overmax_amount': '0.00',
      'patient_pay': '0.00',
      'other_insur': '0.00',
      'net_amount': '0.00'},
     {'procedure_code': 'D0150',
      'billed_amo

In [ ]:
crop_all_eob_tables(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/United_health/mistake_government_pdf_UH/denied__row_mistake/388095333.pdf")


Scanning Page 1
  Page 1 — denial keyword found: False
  patient_tops = []
  item_bottoms = []

Scanning Page 2
  Page 2 — denial keyword found: False
  patient_tops = []
  item_bottoms = []

Scanning Page 3
  Page 3 — denial keyword found: True
  patient_tops = ['204.2', '390.2']
  item_bottoms = ['333.6']
  ✅ Patient@204.2 → ITEM end@333.6
  ⏳ Patient@390.2 → page end (CONTINUING to next page)

Scanning Page 4
  Page 4 — denial keyword found: True
  patient_tops = []
  item_bottoms = []

Scanning Page 5
  Page 5 — denial keyword found: False
  patient_tops = []
  item_bottoms = []

Scanning Page 6
  Page 6 — denial keyword found: True
  patient_tops = []
  item_bottoms = []

Total logical tables detected: 2

--- Logical Table 1 (1 segment(s)) ---


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Cropped page 3: y=204.2→333.6, size=(2550, 539)
  Saved stitched image → EOB_OUTPUT/UHC_Government/388095333/table_1.png


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Expected Rows=2 | Extracted Rows=2

📊 Row Count Validation [Logical Table 1]
----------------------------------------------------------------------
✅ row_count expected=2     | extracted=2     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ billed_amount             computed=895.0      | extracted=895.0      match
✅ allowed_amount            computed=22.0       | extracted=22.0       match
✅ payable_amount            computed=22.0       | extracted=22.0       match
✅ copay_amount              computed=0.0        | extracted=0.0        match
✅ coins_amount              computed=0.0        | extracted=0.0        match
✅ deduct_amount             computed=0.0        | extracted=0.0        match
✅ overmax_amount            computed=0.0        | extracted=0.0        match
✅ patient_pay               computed=0.0        | extracted=0.0        match
✅ other_in

[{'eob_id': '388095333',
  'claim_status': 'Denied',
  'payor': 'UNITED HEALTHCARE - G',
  'confidence_score': 99.5,
  'patients': [{'patient_name': 'ANTONOPOULOS, GEORGE',
    'provider': 'Duc Tang',
    'date_of_service': '05/18/23',
    'services': [{'procedure_code': 'D2740',
      'billed_amount': '$873.00',
      'allowed_amount': '$0.00',
      'payable_amount': '$0.00',
      'copay_amount': '$0.00',
      'coins_amount': '$0.00',
      'deduct_amount': '$0.00',
      'overmax_amount': '$0.00',
      'patient_pay': '$0.00',
      'other_insur': '$0.00',
      'net_amount': '$0.00'},
     {'procedure_code': 'D0220',
      'billed_amount': '$22.00',
      'allowed_amount': '$22.00',
      'payable_amount': '$22.00',
      'copay_amount': '$0.00',
      'coins_amount': '$0.00',
      'deduct_amount': '$0.00',
      'overmax_amount': '$0.00',
      'patient_pay': '$0.00',
      'other_insur': '$0.00',
      'net_amount': '$22.00'}],
    'totals': {'billed_amount': '$895.00',
     '

: 